# Mean and joint fitting in EPDE

Fit the reaction–diffusion equation $u_t = -0.2u + 0.1u_{xx}$ from independent trajectories. Install EPDE with `python -m pip install -e .` from the project root and use that Python environment as the kernel.

## Training and held-out trajectories

Each training record has one spatial mode $k$, so $u_{xx}=-k^2u$ and the two terms cannot be identified separately. The training records have different modes and grid sizes. The held-out record mixes two modes.

Use exact derivatives in the order $u_t$, $u_x$, $u_{xx}$ for `max_deriv_order=(1, 2)`.

In [1]:
import random
from contextlib import redirect_stdout
from io import StringIO

import numpy as np
import epde

def reaction_diffusion(nt, nx, modes):
    t, x = np.meshgrid(np.linspace(0, 2, nt), np.linspace(-np.pi, np.pi, nx), indexing='ij')
    u, ut, ux, uxx = (np.zeros_like(t) for _ in range(4))
    for k, amplitude in modes:
        decay = np.exp(-(0.2 + 0.1 * k**2) * t)
        mode = amplitude * decay * np.sin(k * x)
        u += mode
        ut -= (0.2 + 0.1 * k**2) * mode
        ux += amplitude * k * decay * np.cos(k * x)
        uxx -= k**2 * mode
    return (t, x), u, np.column_stack([ut.ravel(), ux.ravel(), uxx.ravel()])

training = [reaction_diffusion(31, 41, [(1, 1.0)]), reaction_diffusion(23, 53, [(2, 0.7)])]
_, held_out_u, held_out_derivs = reaction_diffusion(29, 47, [(1, 0.8), (3, 0.4)])
holdout = dict(zip(('du/dx0', 'du/dx1', 'd^2u/dx1^2'), held_out_derivs.T))
holdout['u'] = held_out_u.ravel()

## Fit and compare

`mean` averages separate fits. `joint` fits shared coefficients with equal total weight per training record. Both use the same search settings and seed. For each mode, take the front member with the lowest training discrepancy and measure its residual on the held-out record.

In [2]:
def fit(mode):
    random.seed(0)
    np.random.seed(0)
    samples = [(grids, u) for grids, u, _ in training]
    derivatives = [derivs for _, _, derivs in training]
    with redirect_stdout(StringIO()), epde.EpdeMultisample(
        samples, multisample_mode=mode, multisample_weighting='samples',
        system_rps_mode='plain',
        system_rps_params={'target_axes': 0, 'plain_target': True, 'fill_probability': 0.0},
        engine='nsga2', use_solver=False, device='cpu', boundary_width=2,
        equation_terms_max_number=4, equation_factors_max_number=1,
        data_fun_pow=1, deriv_fun_pow=1,
        population_size=8, training_epochs=4, verbose_params={'show_iter_idx': False},
    ) as search:
        search.fit(derivs=derivatives, max_deriv_order=(1, 2))
        equations = search.equations_by_variable(only_print=False)['u']
        equation = min(equations, key=lambda candidate: candidate.fitness_value)
        target = holdout[equation.target.structure[0].label]
        features = [term for term in equation.structure if term is not equation.target]
        prediction = np.full_like(target, equation.weights_final[-1])
        for term, weight in zip(features, equation.weights_final[:-1]):
            prediction += weight * holdout[term.structure[0].label]
        error = np.linalg.norm(prediction - target) / np.linalg.norm(target)
        return equation.text_form, error

for mode in ('mean', 'joint'):
    equation, error = fit(mode)
    print(f'{mode:5s}  {equation}\n       Held-out relative L2 residual: {error:.3g}')

mean   3.027021825193399e-19 * du/dx1{power: 1.0} + -0.4500000000000003 * u{power: 1.0} = du/dx0{power: 1.0}
       Held-out relative L2 residual: 0.554


joint  -0.1999999999999998 * u{power: 1.0} + 0.10000000000000016 * d^2u/dx1^2{power: 1.0} = du/dx0{power: 1.0}
       Held-out relative L2 residual: 8.2e-16
